# Horizontal bar plot of percentage overloaded

## Import packages

In [ ]:
## Time packages
import time
from datetime import datetime

## Memory packages
import psutil # tracking memory and cpu usage
import resource  # tracking memory and cpu usage
import gc
import sys

## Data structure packages
import numpy as np 
import pandas as pd # to create data frames
import pyarrow as pa

from collections import defaultdict

## Math and logic packages
import random
import math

## Load & Save data packages
import os
import glob
import json
import yaml
import joblib

## Plotting packages
import matplotlib.pyplot as plt

## Packages for merging dictionaries
import os
from typing import List, Dict, Any, Iterable, Optional, Tuple, Union
import pandas as pd
import warnings
from joblib import load as joblib_load, dump as joblib_dump

import matplotlib.cm as cm
from matplotlib.patches import Patch
import matplotlib as mpl

## Internal functions packages
from src import figure_ops
from src import input_ops
from src import df_ops
from src import file_ops

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

smart_ds_year = config['smart_ds_years'][0]

## Initialize parameters for saving paths
output_pf_path = config['output_pf_path']
    
# Percent-of-peak range to load (e.g., top 0–10% hours)
start_row_percent = config['start_row_percent']
top_percent_mdh = config['top_percent_mdh']

# File names
transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"

top_n_hours = int(np.ceil(8760*top_percent_mdh/100)) # calculate top city demand hours to run (top_percent_mdh% of hours of the year)

TGW_years_scenarios_ranges = config.get("TGW_years_scenarios_ranges", [])

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")


print(f"\n\nsmart_ds_year: {smart_ds_year}  \n\nstart_row_percent: {start_row_percent} \n\ntop_percent_mdh: {top_percent_mdh} \n\nOutput_pf_path: {output_pf_path}")

print(f"\nfor Option 5 (weather year comparison): {TGW_years_scenarios_ranges}")

## Load merged dictionary w/ summary statistics across years

In [ ]:
# ============================================================
# Load period-level summary dictionaries
# ============================================================

save_folder = "all_regions" 

TGW_scenario = "rcp45hotter"

TGW_weather_year = '2030_2059'

barplot_fig_name = f"figures/overloaded_assets/horizontal_barplot/overloaded_hist_fut_new_esc_{TGW_weather_year}_{TGW_scenario}_{save_folder}.pdf"

# Create the directory structure if it doesn't exist
os.makedirs(os.path.dirname(barplot_fig_name), exist_ok=True)

CITY_REGIONS_TO_RUN = {
    "GSO": ["rural", "industrial", "urban-suburban"],
    "SFO": ["P1U", "P2U", "P1R"],
    "AUS": ["P1U", "P1R", "P2U"],
}


summary_save_dir = os.path.join(
    output_pf_path,
    save_folder,
    "summary_across_weather_years",
    TGW_scenario,
    solar_battery_scenario_folder,
)
transformers_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{transformers_file_name}_summary_across_weather_years.joblib",
)

lines_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{lines_file_name}_summary_across_weather_years.joblib",
)

# Check files exist before loading
for path in [
    transformers_summary_across_years_path,
    lines_summary_across_years_path,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

summary_transformers_dict_weather = joblib.load(
    transformers_summary_across_years_path
)

summary_lines_dict_weather = joblib.load(
    lines_summary_across_years_path
)

print("Loaded:")
print(transformers_summary_across_years_path)
print(lines_summary_across_years_path)

# ============================================================
# Inspect loaded dictionaries
# ============================================================

print("\nTransformer summary dictionary nested keys:")
file_ops.print_nested_keys_structure(summary_transformers_dict_weather)

print("\nTransformer summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    summary_transformers_dict_weather
)

## Process data - merge regions and add MVA

In [ ]:
### Concatenate regions to single city level dataframes (e.g., a single df with all transformers in GSO)
transformers_dict_summary_multi_region_agg_by_city, lines_dict_summary_multi_region_agg_by_city = (
    df_ops.concat_regions_to_city(
        summary_transformers_dict_weather,
        summary_lines_dict_weather,
        TGW_weather_year,
        TGW_scenario,
        smart_ds_year,
        CITY_REGIONS_TO_RUN,
    )
)


## --- Add MVA column ---
def mva_series_lines(df):
    # MVA ≈ NormAmps [A] * Nominal V [kV] / 1000
    if "NormAmps [A]" not in df.columns or "Nominal V [kV]" not in df.columns:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    amps = pd.to_numeric(df["NormAmps [A]"], errors="coerce").fillna(0.0)
    kv   = pd.to_numeric(df["Nominal V [kV]"], errors="coerce").fillna(0.0)
    return (amps * kv) / 1000.0

def mva_series_transformers(df):
    #  kVA/1000 => MVA
    col = "kVA rating [kVA]" if "kVA rating [kVA]" in df.columns else None
    if col is None:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    kva = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    return kva / 1000.0

cities = ["AUS","GSO","SFO"]
for city in cities:
    if 'MVA rating [MVA]' not in transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].columns:
        transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].insert(8, "MVA rating [MVA]",  mva_series_transformers(transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]), allow_duplicates=False)
    if 'MVA rating [MVA]' not in lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].columns:
        lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].insert(9, "MVA rating [MVA]",  mva_series_lines(lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]), allow_duplicates=False)
display(transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].head(2))

## Summarize historical/future overload states and transitions

In [ ]:
# --- Helper functions ---
def below_threshold(column_data, threshold):
    return column_data < threshold
def in_threshold_range(column_data, threshold1, threshold2):
    return (column_data >= threshold1) & (column_data < threshold2)
def above_threshold(column_data, threshold):
    return column_data >= threshold

# Grid reinforcements Thresholds
xfm_cand = 80   # transformers candidate overloading threshold
xfm_crit = 100  # transformers critical overloading threshold
line_cand = 67  # lines candidate overloading threshold
line_crit = 100 # lines critical overloading threshold

# Loading columns
hist_col = "median_annual_max_loading_historical_1990_2019"      # baseline loading column
fut_col  = "median_annual_max_loading_rcp45hotter_2030_2059"     # future loading column

# Initialize dictionaries
lines_overloaded = {(TGW_weather_year, TGW_scenario): {}}
transformers_overloaded = {(TGW_weather_year, TGW_scenario): {}}

# --- Create overloaded data table for transformers ---
cities = ["AUS", "GSO", "SFO"]

for city in cities:
    df = transformers_dict_summary_multi_region_agg_by_city[
        (TGW_weather_year, TGW_scenario)
    ][(smart_ds_year, city)]

    # --- Candidate masks ---
    mask_cand_B_and_F = (
        in_threshold_range(df[hist_col], xfm_cand, xfm_crit) &
        in_threshold_range(df[fut_col],  xfm_cand, xfm_crit))

    mask_cand_B = in_threshold_range(df[hist_col], xfm_cand, xfm_crit)

    mask_cand_F = in_threshold_range(df[fut_col], xfm_cand, xfm_crit)

    mask_cand_B_not_F = (
        in_threshold_range(df[hist_col], xfm_cand, xfm_crit) &
        below_threshold(df[fut_col], xfm_cand))

    mask_cand_F_not_B = (
        in_threshold_range(df[fut_col], xfm_cand, xfm_crit) &
        below_threshold(df[hist_col], xfm_cand))

    # --- Critical masks ---
    mask_crit_B_and_F = (
        above_threshold(df[hist_col], xfm_crit) &
        above_threshold(df[fut_col],  xfm_crit))

    mask_crit_B = above_threshold(df[hist_col], xfm_crit)

    mask_crit_F = above_threshold(df[fut_col], xfm_crit)

    mask_crit_B_not_F = (
        above_threshold(df[hist_col], xfm_crit) &
        below_threshold(df[fut_col], xfm_cand))

    mask_crit_F_not_B = (
        above_threshold(df[fut_col], xfm_crit) &
        below_threshold(df[hist_col], xfm_cand))
    
    # --- Candidate to Critical  masks ---
    mask_cand_B_to_crit_F = mask_cand_B & mask_crit_F
    
    rows = []
    # Add number of assets
    rows.append({
        "Metric": "Cand #",
        "B_and_F": mask_cand_B_and_F.sum(),
        "B":  mask_cand_B.sum(),
        "F":  mask_cand_F.sum(),
        "B_not_F": mask_cand_B_not_F.sum(),
        "F_minus_B":  mask_cand_F.sum() - mask_cand_B.sum(),
        "F_not_B": mask_cand_F_not_B.sum(),
    })
    rows.append({
        "Metric": "Crit #",
        "B_and_F": mask_crit_B_and_F.sum(),
        "B":  mask_crit_B.sum(),
        "F":  mask_crit_F.sum(),
        "B_not_F": mask_crit_B_not_F.sum(),
        "F_minus_B":  mask_crit_F.sum() - mask_crit_B.sum(),
        "F_not_B": mask_crit_F_not_B.sum(),
    })
    rows.append({
        "Metric": "Cand+Crit #",
        "B_and_F": mask_cand_B_and_F.sum() + mask_crit_B_and_F.sum(),
        "B":  mask_cand_B.sum() + mask_crit_B.sum(),
        "F":  mask_cand_F.sum() + mask_crit_F.sum(),
        "B_not_F": mask_crit_B_not_F.sum() + mask_cand_B_not_F.sum(),
        "F_minus_B":  mask_crit_F.sum() + mask_cand_F.sum() - mask_crit_B.sum() - mask_cand_B.sum(),
        "F_not_B": mask_crit_F_not_B.sum() + mask_cand_F_not_B.sum(),
        "F_risk_escalation": mask_crit_F_not_B.sum() + mask_cand_F_not_B.sum() + mask_cand_B_to_crit_F.sum(),
    })
    ## Add MVA values
    rows.append({
        "Metric": "Cand MVA",
        "B_and_F": df.loc[mask_cand_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_cand_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum(),
    })
    rows.append({
        "Metric": "Crit MVA",
        "B_and_F":  df.loc[mask_crit_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":   df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_crit_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum(),
    })
    rows.append({
        "Metric": "Cand+Crit MVA",
        "B_and_F":  df.loc[mask_cand_B_and_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":   df.loc[mask_cand_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_crit_B_not_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum() -  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum(),
        "F_risk_escalation": df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum() + df.loc[mask_cand_B_to_crit_F, 'MVA rating [MVA]'].sum(),        
    })
    df_rows = pd.DataFrame(rows)
    # Round all numeric columns to 2 decimal places (in place)
    numeric = df_rows.select_dtypes(include="number").columns
    df_rows[numeric] = df_rows[numeric].round(2)
    transformers_overloaded[(TGW_weather_year, TGW_scenario)][city] = df_rows
    
    
## --- Create an overloaded data table for lines --- 
cities = ["AUS","GSO","SFO"]
for city in cities:
    df = lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
  
        # --- Candidate masks ---
    mask_cand_B_and_F = (
        in_threshold_range(df[hist_col], line_cand, line_crit) &
        in_threshold_range(df[fut_col],  line_cand, line_crit))

    mask_cand_B = in_threshold_range(df[hist_col], line_cand, line_crit)

    mask_cand_F = in_threshold_range(df[fut_col], line_cand, line_crit)

    mask_cand_B_not_F = (
        in_threshold_range(df[hist_col], line_cand, line_crit) &
        below_threshold(df[fut_col], line_cand))

    mask_cand_F_not_B = (
        in_threshold_range(df[fut_col], line_cand, line_crit) &
        below_threshold(df[hist_col], line_cand))

    # --- Critical masks ---
    mask_crit_B_and_F = (
        above_threshold(df[hist_col], line_crit) &
        above_threshold(df[fut_col],  line_crit))

    mask_crit_B = above_threshold(df[hist_col], line_crit)

    mask_crit_F = above_threshold(df[fut_col], line_crit)

    mask_crit_B_not_F = (
        above_threshold(df[hist_col], line_crit) &
        below_threshold(df[fut_col], line_cand))

    mask_crit_F_not_B = (
        above_threshold(df[fut_col], line_crit) &
        below_threshold(df[hist_col], line_cand))
    
    # --- Candidate to Critical  masks ---
    mask_cand_B_to_crit_F = mask_cand_B & mask_crit_F
    
    rows = []
    
    # Add number of assets
    rows.append({
        "Metric": "Cand #",
        "B_and_F": mask_cand_B_and_F.sum(),
        "B":  mask_cand_B.sum(),
        "F":  mask_cand_F.sum(),
        "B_not_F": mask_cand_B_not_F.sum(),
        "F_minus_B":  mask_cand_F.sum() - mask_cand_B.sum(),
        "F_not_B": mask_cand_F_not_B.sum(),
    })
    rows.append({
        "Metric": "Crit #",
        "B_and_F": mask_crit_B_and_F.sum(),
        "B":  mask_crit_B.sum(),
        "F":  mask_crit_F.sum(),
        "B_not_F": mask_crit_B_not_F.sum(),
        "F_minus_B":  mask_crit_F.sum() - mask_crit_B.sum(),
        "F_not_B": mask_crit_F_not_B.sum(),
    })
    rows.append({
        "Metric": "Cand+Crit #",
        "B_and_F": mask_cand_B_and_F.sum() + mask_crit_B_and_F.sum(),
        "B":  mask_cand_B.sum() + mask_crit_B.sum(),
        "F":  mask_cand_F.sum() + mask_crit_F.sum(),
        "B_not_F": mask_crit_B_not_F.sum() + mask_cand_B_not_F.sum(),
        "F_minus_B":  mask_crit_F.sum() + mask_cand_F.sum() - mask_crit_B.sum() - mask_cand_B.sum(),
        "F_not_B": mask_crit_F_not_B.sum() + mask_cand_F_not_B.sum(),
        "F_risk_escalation": mask_crit_F_not_B.sum() + mask_cand_F_not_B.sum() + mask_cand_B_to_crit_F.sum(),
    })
    ## Add MVA values
    rows.append({
        "Metric": "Cand MVA",
        "B_and_F": df.loc[mask_cand_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_cand_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum(),
    })
    rows.append({
        "Metric": "Crit MVA",
        "B_and_F":  df.loc[mask_crit_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":   df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_crit_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum(),
    })
    rows.append({
        "Metric": "Cand+Crit MVA",
        "B_and_F":  df.loc[mask_cand_B_and_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_B_and_F, 'MVA rating [MVA]'].sum(),
        "B":   df.loc[mask_cand_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum(),
        "F":   df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_crit_F, 'MVA rating [MVA]'].sum(),
        "B_not_F":  df.loc[mask_crit_B_not_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_B_not_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B":   df.loc[mask_crit_F, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F, 'MVA rating [MVA]'].sum() -  df.loc[mask_crit_B, 'MVA rating [MVA]'].sum() -  df.loc[mask_cand_B, 'MVA rating [MVA]'].sum(),
        "F_not_B":  df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum(),
        "F_risk_escalation": df.loc[mask_crit_F_not_B, 'MVA rating [MVA]'].sum() +  df.loc[mask_cand_F_not_B, 'MVA rating [MVA]'].sum() + df.loc[mask_cand_B_to_crit_F, 'MVA rating [MVA]'].sum(),
    })
    df_rows = pd.DataFrame(rows)
    # Round all numeric columns to 2 decimal places (in place)
    numeric = df_rows.select_dtypes(include="number").columns
    df_rows[numeric] = df_rows[numeric].round(2)
    lines_overloaded[(TGW_weather_year, TGW_scenario)][city] = df_rows
    
display(transformers_overloaded[(TGW_weather_year, TGW_scenario)][city])

## Convert counts and capacities to shares

In [ ]:
def add_percentage_columns_by_row_ranges(
    df,
    total1=None, total1_rows=None,   # e.g., (1, 3)  -> rows 1..3 inclusive
    total2=None, total2_rows=None,   # e.g., (4, 6)  -> rows 4..6 inclusive
    start_col=1,                     # first numeric column index
    n_numeric=5,                     # how many numeric columns to convert
    suffix="_pct",
    decimals=2,
):
    """
    Adds percentage columns for numeric columns [start_col : start_col+n_numeric).
    percentage = 100 * value / row_total

    row_total is assigned by 1-based, inclusive row ranges:
      - rows in total1_rows -> total1
      - rows in total2_rows -> total2
      - all others -> NaN (percentages become NaN)

    Notes:
      * If a range is None or a total is None/0, that assignment is skipped.
      * If ranges overlap, later assignments take precedence.
    """
    num_cols = df.columns[start_col:start_col + n_numeric]
    nrows = len(df)
    pos = np.arange(1, nrows + 1)   # 1-based positions

    total_per_row = pd.Series(np.nan, index=df.index, dtype="float64")

    def apply_range(total, rng):
        if total is None or not rng or len(rng) != 2:
            return
        lo, hi = int(rng[0]), int(rng[1])
        if total == 0:
            return  # avoid div-by-zero (leave NaN)
        # clamp to valid bounds; inclusive
        lo = max(1, lo); hi = min(nrows, hi)
        if lo <= hi:
            mask = (pos >= lo) & (pos <= hi)
            total_per_row.iloc[mask] = float(total)

    apply_range(total1, total1_rows)
    apply_range(total2, total2_rows)

    pct = (
        df[num_cols].apply(pd.to_numeric, errors="coerce")
        .div(total_per_row, axis=0)
        .mul(100.0)
        .round(decimals)
    )
    pct.columns = [f"{c}{suffix}" for c in num_cols]
    return pd.concat([df, pct], axis=1)


cities = ["AUS","GSO","SFO"]
for city in cities:
    df_data = transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    df_overloaded = transformers_overloaded[(TGW_weather_year, TGW_scenario)][city]
    transformers_overloaded[(TGW_weather_year, TGW_scenario)][city] = add_percentage_columns_by_row_ranges(df_overloaded,total1=len(df_data), total1_rows=(1, 3), total2=df_data['MVA rating [MVA]'].sum(),total2_rows=(4, 6),start_col=1,n_numeric=7, suffix="_pct",decimals=2,)
    
    df_data = lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    df_overloaded = lines_overloaded[(TGW_weather_year, TGW_scenario)][city]
    lines_overloaded[(TGW_weather_year, TGW_scenario)][city] = add_percentage_columns_by_row_ranges(df_overloaded,total1=len(df_data), total1_rows=(1, 3), total2=df_data['MVA rating [MVA]'].sum(),total2_rows=(4, 6),start_col=1,n_numeric=7, suffix="_pct",decimals=2,)
    
    
city = 'AUS'

print("Transformers:\n")
df_data = transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
display(transformers_overloaded[(TGW_weather_year, TGW_scenario)][city])
display(f"Total # of transformers in {city}: {len(df_data)}")
display(f"Total capacity of transformers in {city}: {df_data['MVA rating [MVA]'].sum()} MVA")


print("Lines:\n")

df_data = lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
display(lines_overloaded[(TGW_weather_year, TGW_scenario)][city])
display(f"Total # of lines in {city}: {len(df_data)}")
display(f"Total capacity of lines in {city}: {df_data['MVA rating [MVA]'].sum()} MVA")

## Plot Share of overloaded assets

In [ ]:
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams["text.usetex"] = True

fontsize = 14

plt.rcParams.update({
    "font.size": fontsize,
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize,
})

def plot_overload_state_transition_tx_ln(
    transformers_overloaded,
    lines_overloaded,
    TGW_weather_year="2058",
    TGW_scenario="rcp45hotter",
    cities=("AUS", "GSO", "SFO"),

    state_labels=("Historical", "Future"),
    transition_labels=(
        "Newly at risk\n(Safe $\\rightarrow$ At-risk/Critical)",
        "Increased risk\n(Safe $\\rightarrow$ At-risk/Critical\n+ At-risk $\\rightarrow$ Critical)",
    ),

    x_label_tx="Share of overloaded \ntransformers [\\%]",
    x_label_ln="Share of overloaded \npower lines [\\%]",

    xlim_tx=None,
    xlim_ln=None,

    # If True, subplots in each column share an x-axis.
    # If False, each subplot has its own x-axis.
    sharex=True,

    # Order of the two portions within each horizontal bar:
    #   "at_risk_first"  -> At-risk on left, Critical on right
    #   "critical_first" -> Critical on left, At-risk on right
    bar_segment_order="at_risk_first",

    subplot_frame_color="black",
    subplot_frame_linewidth=0.8,

    legend_loc="upper center",
    legend_bbox_to_anchor=(0.62, 1.01),

    value_decimals=1,
    figsize=(12, 9),
    savepath=None,
):
    """
    3x2 figure:
      rows = cities
      left = transformers
      right = power lines

    Within each subplot:
      SYSTEM STATE
          Historical
          Future

      CLIMATE-DRIVEN TRANSITIONS
          Newly at risk
          Increased risk

    Critical portions are hatched.
    At-risk portions are solid.
    """

    if bar_segment_order not in ("at_risk_first", "critical_first"):
        raise ValueError(
            "bar_segment_order must be either "
            "'at_risk_first' or 'critical_first'."
        )

    key = (TGW_weather_year, TGW_scenario)

    fig, axes = plt.subplots(
        3,
        2,
        figsize=figsize,
        sharex="col" if sharex else False,
        sharey=True,
    )

    colors = {
        "Historical": "#7A7A7A",
        "Future": "#4C72B0",
        "Newly at risk": "#DD8452",
        "Increased risk": "#C44E52",
    }

    hatch_crit = "///"
    bar_height = 0.62

    # Top-to-bottom positions.
    # Gap between system state and transition metrics.
    y_positions = [4.0, 3.0, 1.5, 0.5]

    y_labels = [
        state_labels[0],
        state_labels[1],
        transition_labels[0],
        transition_labels[1],
    ]

    # ==================================================================
    # collect all values used in the figure
    # ==================================================================
    figure_data_rows = []

    def draw_stacked_barh(ax, y, cand, crit, color):

        if bar_segment_order == "at_risk_first":

            # At-risk
            ax.barh(
                y,
                cand,
                height=bar_height,
                left=0,
                color=color,
                edgecolor="black",
                linewidth=0.7,
            )

            # Critical
            ax.barh(
                y,
                crit,
                height=bar_height,
                left=cand,
                color=color,
                edgecolor="black",
                linewidth=0.7,
                hatch=hatch_crit,
            )

        elif bar_segment_order == "critical_first":

            # Critical
            ax.barh(
                y,
                crit,
                height=bar_height,
                left=0,
                color=color,
                edgecolor="black",
                linewidth=0.7,
                hatch=hatch_crit,
            )

            # At-risk
            ax.barh(
                y,
                cand,
                height=bar_height,
                left=crit,
                color=color,
                edgecolor="black",
                linewidth=0.7,
            )

    def draw_city(
        ax,
        df_city,
        xlim,
        city,
        component,
    ):

        r_tot = _get_metric_row(df_city, "Cand+Crit #")
        r_cand = _get_metric_row(df_city, "Cand #")
        r_crit = _get_metric_row(df_city, "Crit #")

        # ----------------------------------------------------------
        # Historical
        # ----------------------------------------------------------
        B_tot = float(r_tot["B_pct"])
        B_cand = float(r_cand["B_pct"])
        B_crit = float(r_crit["B_pct"])

        # ----------------------------------------------------------
        # Future
        # ----------------------------------------------------------
        F_tot = float(r_tot["F_pct"])
        F_cand = float(r_cand["F_pct"])
        F_crit = float(r_crit["F_pct"])

        # ----------------------------------------------------------
        # Newly at risk:
        # Safe -> At-risk/Critical
        # ----------------------------------------------------------
        new_cand = float(r_cand["F_not_B_pct"])
        new_crit = float(r_crit["F_not_B_pct"])
        new_tot = new_cand + new_crit

        # ----------------------------------------------------------
        # Increased risk:
        # Safe -> At-risk/Critical + At-risk -> Critical
        # ----------------------------------------------------------
        esc_tot = float(r_tot["F_risk_escalation_pct"])

        at_risk_to_critical = esc_tot - new_tot

        if (
            at_risk_to_critical < 0
            and abs(at_risk_to_critical) < 0.05
        ):
            at_risk_to_critical = 0.0

        esc_cand = new_cand
        esc_crit = new_crit + at_risk_to_critical

        city_name = CITY_MAP.get(city, city)

        figure_data_rows.extend([
            {
                "Weather year": TGW_weather_year,
                "Climate scenario": TGW_scenario,
                "City": city_name,
                "Component": component,
                "Metric": state_labels[0],
                "At-risk [%]": B_cand,
                "Critical [%]": B_crit,
                "Total [%]": B_tot,
            },
            {
                "Weather year": TGW_weather_year,
                "Climate scenario": TGW_scenario,
                "City": city_name,
                "Component": component,
                "Metric": state_labels[1],
                "At-risk [%]": F_cand,
                "Critical [%]": F_crit,
                "Total [%]": F_tot,
            },
            {
                "Weather year": TGW_weather_year,
                "Climate scenario": TGW_scenario,
                "City": city_name,
                "Component": component,
                "Metric": transition_labels[0],
                "At-risk [%]": new_cand,
                "Critical [%]": new_crit,
                "Total [%]": new_tot,
            },
            {
                "Weather year": TGW_weather_year,
                "Climate scenario": TGW_scenario,
                "City": city_name,
                "Component": component,
                "Metric": transition_labels[1],
                "At-risk [%]": esc_cand,
                "Critical [%]": esc_crit,
                "Total [%]": esc_tot,
            },
        ])

        # ----------------------------------------------------------
        # Plot
        # ----------------------------------------------------------
        draw_stacked_barh(
            ax,
            y_positions[0],
            B_cand,
            B_crit,
            colors["Historical"],
        )

        draw_stacked_barh(
            ax,
            y_positions[1],
            F_cand,
            F_crit,
            colors["Future"],
        )

        draw_stacked_barh(
            ax,
            y_positions[2],
            new_cand,
            new_crit,
            colors["Newly at risk"],
        )

        draw_stacked_barh(
            ax,
            y_positions[3],
            esc_cand,
            esc_crit,
            colors["Increased risk"],
        )

        totals = [
            B_tot,
            F_tot,
            new_tot,
            esc_tot,
        ]

        # ----------------------------------------------------------
        # X-axis
        # ----------------------------------------------------------
        if xlim is not None:
            ax.set_xlim(*xlim)
            x_range = xlim[1] - xlim[0]
        else:
            xmax = max(totals)
            headroom = max(0.5, 0.12 * xmax)
            ax.set_xlim(0, xmax + headroom)
            x_range = xmax + headroom

        offset = 0.012 * x_range

        for y, total in zip(y_positions, totals):
            ax.text(
                total + offset,
                y,
                f"{total:.{value_decimals}f}\\%",
                ha="left",
                va="center",
            )

        # ----------------------------------------------------------
        # Separate states from transitions
        # ----------------------------------------------------------
        ax.axhline(
            2.25,
            color="0.75",
            linewidth=0.8,
        )

        # ----------------------------------------------------------
        # Formatting
        # ----------------------------------------------------------
        ax.set_yticks(y_positions)
        ax.set_yticklabels(y_labels)

        ax.xaxis.grid(
            True,
            linestyle="--",
            linewidth=0.5,
            alpha=0.6,
        )

        ax.yaxis.grid(False)

        for spine in ax.spines.values():
            spine.set_color(subplot_frame_color)
            spine.set_linewidth(subplot_frame_linewidth)

    # ==============================================================
    # Draw cities
    # ==============================================================
    for i, city in enumerate(cities):

        city_name = CITY_MAP.get(city, city)

        draw_city(
            axes[i, 0],
            transformers_overloaded[key][city],
            xlim_tx,
            city=city,
            component="Transformers",
        )

        draw_city(
            axes[i, 1],
            lines_overloaded[key][city],
            xlim_ln,
            city=city,
            component="Power lines",
        )

        axes[i, 0].set_title(
            city_name,
            loc="left",
            fontweight="bold",
        )

        axes[i, 1].tick_params(
            axis="y",
            labelleft=False,
        )

    # ==============================================================
    # Column headers
    # ==============================================================
    axes[0, 0].text(
        0.5,
        1.12,
        "Transformers",
        transform=axes[0, 0].transAxes,
        ha="center",
        fontweight="bold",
    )

    axes[0, 1].text(
        0.5,
        1.12,
        "Power lines",
        transform=axes[0, 1].transAxes,
        ha="center",
        fontweight="bold",
    )

    # ==============================================================
    # X-axis labels
    # ==============================================================
    if sharex:

        axes[-1, 0].set_xlabel(x_label_tx)
        axes[-1, 1].set_xlabel(x_label_ln)

    else:

        for i in range(len(cities)):

            axes[i, 0].set_xlabel(x_label_tx)
            axes[i, 1].set_xlabel(x_label_ln)

            axes[i, 0].tick_params(
                axis="x",
                labelbottom=True,
            )

            axes[i, 1].tick_params(
                axis="x",
                labelbottom=True,
            )

    # ==============================================================
    # Legend
    # ==============================================================
    legend_handles = [
        Patch(
            facecolor="white",
            edgecolor="black",
            label="At-risk",
        ),
        Patch(
            facecolor="white",
            edgecolor="black",
            hatch=hatch_crit,
            label="Critical",
        ),
    ]

    if bar_segment_order == "critical_first":
        legend_handles = legend_handles[::-1]

    fig.legend(
        handles=legend_handles,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox_to_anchor,
        ncol=2,
        frameon=False,
    )

    # ==============================================================
    # Panel labels
    # ==============================================================
    panel_labels = ["a", "b", "c", "d", "e", "f"]

    k = 0

    for i in range(3):
        for j in range(2):

            axes[i, j].text(
                -0.08,
                1.05,
                panel_labels[k],
                transform=axes[i, j].transAxes,
                fontweight="bold",
                ha="left",
                va="top",
            )

            k += 1

    plt.tight_layout()

    if savepath:
        plt.savefig(
            savepath,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.4,
        )

    plt.show()

    # ==================================================================
    #  return data used by the figure
    # ==================================================================
    figure_data = pd.DataFrame(figure_data_rows)

    return figure_data

figure_data = plot_overload_state_transition_tx_ln(
    transformers_overloaded=transformers_overloaded,
    lines_overloaded=lines_overloaded,

    TGW_weather_year=TGW_weather_year,
    TGW_scenario=TGW_scenario,

    cities=("AUS", "GSO", "SFO"),

    state_labels=(
        "Historical",
        "Future",
    ),

    transition_labels=(
        "Newly overloaded: Safe $\\rightarrow$ At-risk",
        "Increased risk: Safe $\\rightarrow$ At-risk\nand At-risk $\\rightarrow$ Critical",
    ),

    x_label_tx="Share of overloaded transformers [\\%]",
    x_label_ln="Share of overloaded power lines [\\%]",

    xlim_tx=(0, 35),
    xlim_ln=(0, 5),

    # Give every subplot its own x-axis
    sharex=False,

    # Put Critical first (left side), At-risk second (right side)
    bar_segment_order="critical_first",

    subplot_frame_color="black",
    subplot_frame_linewidth=0.8,

    legend_loc="upper center",
    legend_bbox_to_anchor=(0.6, 1.01),

    value_decimals=1,

    figsize=(12, 9), # (12,9)

    savepath=barplot_fig_name,
)

display(figure_data)